In [22]:
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [23]:
spotify = pd.read_csv("spotify_merged_clean.csv")

continuous_features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

spotify.drop(columns=['key', 'mode'], inplace=True)

df = spotify.copy()

In [24]:

# ------------------------------------------------------------------
# Configuration: adjust these names to match your dataframe
# ------------------------------------------------------------------

MACRO_COL = "macro_genre"
SUBGENRE_COL = "genre"       # change to "subgenre" if needed

FEATURES = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

MACRO_GENRES = [
    "Electronic",
    "Rock",
    "Pop",
    "Latin",
]


# ------------------------------------------------------------------
# Keep only tracks with a genuine granular genre annotation
# ------------------------------------------------------------------

df_granular = df.loc[
    df[SUBGENRE_COL].notna()
    & df[MACRO_COL].notna()
    & (df[SUBGENRE_COL].astype(str).str.strip() != "")
    & (
        df[SUBGENRE_COL].astype(str).str.lower().str.strip()
        != df[MACRO_COL].astype(str).str.lower().str.strip()
    )
].copy()

# Remove rows with missing audio features
df_granular = df_granular.dropna(subset=FEATURES).reset_index(drop=True)

# Standardize once on the complete granular subset.
# This preserves a common feature scale across macro-genres.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_granular[FEATURES])

# Store standardized vectors in a separate dataframe
X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=FEATURES,
    index=df_granular.index
)

print(f"Tracks with granular annotations: {len(df_granular):,}")
print("\nTracks by macro-genre:")
display(
    df_granular[MACRO_COL]
    .value_counts()
    .loc[lambda x: x.index.isin(MACRO_GENRES)]
    .to_frame("tracks")
)

Tracks with granular annotations: 90,658

Tracks by macro-genre:


,tracks
macro_genre,
Electronic,18673
Latin,7138
Pop,4787
Rock,4483


In [25]:
from sklearn.metrics.pairwise import cosine_distances

def mean_group_dispersion(X, labels, min_group_size=2):
    """
    For each group, compute the mean Euclidean distance of its tracks
    from the group centroid in standardized feature space.

    Returns the unweighted mean across valid groups.
    """
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    group_scores = []

    for label in pd.unique(labels):
        mask = labels == label
        X_group = X[mask]

        if len(X_group) < min_group_size:
            continue

        centroid = X_group.mean(axis=0, keepdims=True)

        distances = cosine_distances(
            X_group,
            centroid
        ).ravel()

        group_scores.append(distances.mean())

    return float(np.mean(group_scores)) if group_scores else np.nan


def local_community_consistency(G, partition):
    """
    Average proportion of each node's neighbours belonging to the
    same Louvain community as the node itself.
    """
    scores = []

    for node in G.nodes:
        neighbours = list(G.neighbors(node))

        if not neighbours:
            continue

        same_community = sum(
            partition[neighbour] == partition[node]
            for neighbour in neighbours
        )

        scores.append(same_community / len(neighbours))

    return float(np.mean(scores)) if scores else np.nan


def evaluate_macro_network(
    df_macro,
    X_macro,
    subgenre_col,
    k=5,
    resolution=1.0,
    random_state=42
):
    """
    Build a symmetrized cosine k-NN graph and compute:
      - subgenre assortativity
      - Louvain modularity
      - local community consistency
      - subgenre acoustic dispersion
      - community acoustic dispersion
    """
    df_macro = df_macro.reset_index(drop=True)
    X_macro = np.asarray(X_macro, dtype=float)

    n_tracks = len(df_macro)

    if n_tracks < 3:
        raise ValueError("At least three tracks are required.")

    # +1 because the nearest neighbour returned is the observation itself
    k_effective = min(k, n_tracks - 1)

    neighbours = NearestNeighbors(
        n_neighbors=k_effective + 1,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1
    )

    neighbours.fit(X_macro)
    distances, indices = neighbours.kneighbors(X_macro)

    # --------------------------------------------------------------
    # Build an undirected graph from the directed k-NN relationships
    # --------------------------------------------------------------

    G = nx.Graph()
    G.add_nodes_from(range(n_tracks))

    for source in range(n_tracks):
        for distance, target in zip(
            distances[source, 1:],
            indices[source, 1:]
        ):
            similarity = max(0.0, 1.0 - float(distance))

            # If an edge is selected in both directions, retain the
            # highest observed similarity.
            if G.has_edge(source, target):
                G[source][target]["weight"] = max(
                    G[source][target]["weight"],
                    similarity
                )
            else:
                G.add_edge(
                    source,
                    int(target),
                    weight=similarity,
                    distance=float(distance)
                )

    # Add subgenre labels as node attributes
    subgenre_labels = df_macro[subgenre_col].astype(str).to_dict()
    nx.set_node_attributes(G, subgenre_labels, "subgenre")

    # --------------------------------------------------------------
    # Louvain community detection
    # --------------------------------------------------------------

    communities = nx.community.louvain_communities(
        G,
        weight="weight",
        resolution=resolution,
        seed=random_state
    )

    partition = {
        node: community_id
        for community_id, community_nodes in enumerate(communities)
        for node in community_nodes
    }

    community_labels = np.array(
        [partition[node] for node in range(n_tracks)]
    )

    # --------------------------------------------------------------
    # Metrics
    # --------------------------------------------------------------

    assortativity = nx.attribute_assortativity_coefficient(
        G,
        "subgenre"
    )

    modularity = nx.community.modularity(
        G,
        communities,
        weight="weight",
        resolution=resolution
    )

    consistency = local_community_consistency(G, partition)

    subgenre_dispersion = mean_group_dispersion(
        X_macro,
        df_macro[subgenre_col].astype(str).to_numpy()
    )

    community_dispersion = mean_group_dispersion(
        X_macro,
        community_labels
    )

    result = {
        "Tracks": n_tracks,
        "Edges": G.number_of_edges(),
        "Communities": len(communities),
        "Subgenre assortativity": assortativity,
        "Modularity": modularity,
        "Local consistency": consistency,
        "Subgenre dispersion": subgenre_dispersion,
        "Community dispersion": community_dispersion
    }

    return result, G, partition



In [26]:
for macro_genre in MACRO_GENRES:
    vc = spotify['genre'][spotify['macro_genre'] == macro_genre].value_counts()

    print(vc)

genre
edm                  4926
idm                   958
chicago-house         956
breakbeat             955
club                  946
detroit-techno        920
drum-and-bass         915
deep-house            915
hardstyle             880
garage                868
minimal-techno        845
industrial            809
electronic            808
disco                 769
progressive-house     677
trance                676
dance                 486
techno                401
electro               384
dubstep               253
house                 134
Name: count, dtype: int64
genre
rock           3453
alt-rock        798
rock-n-roll     773
grunge          744
hard-rock       626
psych-rock      550
rockabilly      543
j-rock          449
Name: count, dtype: int64
genre
pop          4644
cantopop      955
k-pop         854
mandopop      843
power-pop     833
synth-pop     708
j-pop         594
Name: count, dtype: int64
genre
latin        4130
forro         968
tango         935
brazil      

In [27]:
# ------------------------------------------------------------------
# Run the same pipeline using different hardcoded filters
# ------------------------------------------------------------------

SUBSET_FILTERS = {
    "Electronic": (
        df_granular[MACRO_COL].eq("Electronic")
        & df_granular[SUBGENRE_COL].isin([
            "breakbeat",
            "chicago-house",
            "club",
            "dance",
            "deep-house",
            "detroit-techno",
            "disco",
            "drum-and-bass",
            "dubstep",
            "edm",
            "electro",
            "garage",
            "hardstyle",
            "house",
            "idm",
            "industrial",
            "minimal-techno",
            "progressive-house",
            "techno",
            "trance"
        ])
    ),

    "Rock": (
        df_granular[MACRO_COL].eq("Rock")
        & df_granular[SUBGENRE_COL].isin([
            "alt-rock",
            "rock-n-roll",
            "grunge",
            "hard-rock",
            "psych-rock",
            "rockabilly",
            "j-rock",
        ])
    ),

    "Pop": (
        df_granular[MACRO_COL].eq("Pop")
        & df_granular[SUBGENRE_COL].isin([
            "cantopop",
            "j-pop",
            "k-pop",
            "mandopop",
            "power-pop",
            "synth-pop"
        ])
    ),

    "Latin": (
        df_granular[MACRO_COL].eq("Latin")
        & df_granular[SUBGENRE_COL].isin([
            "brazil",
            "forro",
            "mpb",
            "pagode",
            "reggaeton",
            "salsa",
            "samba",
            "sertanejo",
            "tango"
        ])
    ),
}


In [28]:

# ------------------------------------------------------------------
# Run the same pipeline for every selected macro-genre
# ------------------------------------------------------------------

results = []
network_objects = {}

for macro in MACRO_GENRES:

    mask = SUBSET_FILTERS[macro]

    df_macro = df_granular.loc[mask].copy()
    X_macro = X_scaled_df.loc[mask].to_numpy()

    if len(df_macro) < 3:
        print(f"Skipping {macro}: insufficient observations.")
        continue

    metrics, G_macro, partition_macro = evaluate_macro_network(
        df_macro=df_macro,
        X_macro=X_macro,
        subgenre_col=SUBGENRE_COL,
        k=5,                  # use the same final k as your main analysis
        resolution=1.0,
        random_state=42
    )

    metrics[MACRO_COL] = macro
    results.append(metrics)

    # Retain graphs and partitions for later inspection if needed
    network_objects[macro] = {
        "graph": G_macro,
        "partition": partition_macro,
        "data": df_macro.reset_index(drop=True),
        "X": X_macro
    }

results_df = (
    pd.DataFrame(results)
    .set_index(MACRO_COL)
    .loc[MACRO_GENRES]
)

metric_columns = [
    "Subgenre assortativity",
    "Modularity",
    "Local consistency",
    "Subgenre dispersion",
    "Community dispersion"
]

display(
    results_df[
        ["Tracks", "Edges", "Communities"] + metric_columns
    ].round(3)
)

,Tracks,Edges,Communities,Subgenre assortativity,Modularity,Local consistency,Subgenre dispersion,Community dispersion
macro_genre,,,,,,,,
Electronic,18673,67105,23,0.199,0.820,0.860,0.423,0.176
Rock,4483,15735,23,0.180,0.791,0.828,0.520,0.171
Pop,4787,16869,21,0.258,0.786,0.828,0.520,0.200
Latin,6806,24094,20,0.336,0.797,0.841,0.438,0.175
